In [ ]:
import os, re, csv, time, math, copy, random, hashlib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc as sk_auc
import matplotlib.pyplot as plt

import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset

torch.set_grad_enabled(True)

# -------------------------
# REPRO
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


# New cell by cell code

In [ ]:
# =========================================
# CELL 1 — INSTALLS (run once)
# =========================================
!pip -q install -U "qiskit>=1.0" qiskit-aer qiskit-machine-learning transformers safetensors

# If torch is broken in your runtime, uncomment:
# !pip -q uninstall -y torch torchvision torchaudio
# !pip -q install -U --no-cache-dir torch torchvision torchaudio


In [ ]:
# =========================================
# CELL 2 — IMPORTS + CONFIG
# =========================================
import os, re, math, copy, time, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# -----------------------
# PATHS (EDIT THESE)
# -----------------------
RESNET_CKPT = "/kaggle/input/resnet50/pytorch/default/1/best_resnet50.pt"

NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

# This is YOUR NIH test CSV like screenshot (has y_* and/or cnn_* columns)
TARGET_TEACHER_CSV = "/kaggle/input/llm-files/nih_test.csv"

# Optional: if you have a CheX reference set for Wasserstein stats
CHEX_REF_IMG  = ""   # folder root
CHEX_REF_LIST = ""   # txt list of image relative paths
CHEX_REF_MAX  = 6000

# Canonical label set (must match everywhere)
LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]
C = len(LABELS)

BATCH_SIZE  = 16 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0

# TTDA knobs
USE_TTA = True
ENT_THR = 0.55
TENT_LR = 1e-4
COTTA_LR = 1e-4
WASS_LR = 1e-4

# Distillation knobs (from your LLM CSV)
USE_TEACHER = True
TEACH_W_LLM  = 0.65        # blend y_* vs cnn_*
LLM_CONF_THR = 0.20        # trust y_* only when |y-0.5| > thr
BETA_DISTILL = 1.0

# PQC knobs
RUN_PQC = True
QUBITS_LIST = [6]
N_ENT_LAYERS = 2
PQC_LOGIT_SCALE = 0.20
LR_PQC = 3e-4
MAX_PQC_STEPS = 250        # keep small (qiskit slow)
FID_WEIGHT = 0.15

for p in [RESNET_CKPT, NIH_IMG, NIH_CSV, NIH_TEST, NIH_STREAM, TARGET_TEACHER_CSV]:
    print(("✅" if os.path.exists(p) else "❌"), p)


In [ ]:
# =========================================
# CELL 3 — DATA LOADING (NIH GT)
# =========================================
def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def nih_resolver(rel):
    p = os.path.join(NIH_IMG, rel)
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    return p

def build_nih_df(list_txt, nih_csv):
    wanted = set(read_txt_lines(list_txt))
    df = pd.read_csv(nih_csv)
    df = df[df["Image Index"].isin(wanted)].copy()

    findings = df["Finding Labels"].fillna("")

    def has(lbl):
        return findings.apply(lambda s: int(lbl in str(s).split("|")))

    # NIH uses "Effusion" -> map to our canonical "Pleural Effusion"
    df["Atelectasis"]      = has("Atelectasis")
    df["Cardiomegaly"]     = has("Cardiomegaly")
    df["Consolidation"]    = has("Consolidation")
    df["Edema"]            = has("Edema")
    df["Pleural Effusion"] = has("Effusion")

    df = df.rename(columns={"Image Index":"image"})
    keep = ["image"] + LABELS
    return df[keep].reset_index(drop=True)

class CXRDataset(Dataset):
    def __init__(self, df, resolver, labeled=True):
        self.df = df.reset_index(drop=True)
        self.resolver = resolver
        self.labeled = labeled
        self.images = self.df["image"].tolist()
        if labeled:
            self.Y = self.df[LABELS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_path = self.resolver(self.images[idx])
        img = Image.open(img_path).convert("RGB")
        meta = {
            "image": self.images[idx],
            "image_key": os.path.basename(self.images[idx]),
        }
        y = torch.from_numpy(self.Y[idx]) if self.labeled else torch.zeros(C, dtype=torch.float32)
        return img, y, meta

def pil_collate(batch):
    imgs, ys, metas = zip(*batch)
    return list(imgs), torch.stack(list(ys)), list(metas)

nih_test_df = build_nih_df(NIH_TEST, NIH_CSV)
nih_stream_df = build_nih_df(NIH_STREAM, NIH_CSV)

test_loader = DataLoader(CXRDataset(nih_test_df, nih_resolver, labeled=True),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         collate_fn=pil_collate)

stream_loader = DataLoader(CXRDataset(nih_stream_df, nih_resolver, labeled=False),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                           collate_fn=pil_collate)

print("NIH test:", len(nih_test_df), "NIH stream:", len(nih_stream_df))


In [ ]:
# =========================================
# CELL 4 — PREPROCESS (match training as closely as possible)
# =========================================
# If your training used different mean/std, change here.
norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm])

try:
    autoaug = T.AutoAugment(policy=T.AutoAugmentPolicy.IMAGENET)
    autoaug_tf = T.Compose([T.Resize(256), T.CenterCrop(224), autoaug, T.ToTensor(), norm])
except Exception:
    autoaug_tf = None

def batch_tf(pils, tf):
    return torch.stack([tf(im) for im in pils], dim=0)


In [ ]:
# =========================================
# CELL 5 — SAFE CHECKPOINT LOAD + MODEL (fixes random head problem)
# =========================================
def strip_prefix(sd, prefixes=("module.","model.","backbone.","net.")):
    out = {}
    for k,v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p):
                kk = kk[len(p):]
        out[kk]=v
    return out

def load_state_dict_any(path):
    # If your environment blocks torch.load for security, convert to safetensors.
    ckpt = torch.load(path, map_location="cpu", weights_only=True) if "weights_only" in torch.load.__code__.co_varnames else torch.load(path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt: sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt: sd = ckpt["model"]
    else: sd = ckpt
    return strip_prefix(sd)

# Build standard ResNet50 classifier
net = models.resnet50(weights=None)
net.fc = nn.Linear(2048, C)

sd = load_state_dict_any(RESNET_CKPT)
missing, unexpected = net.load_state_dict(sd, strict=False)
print("Loaded ckpt strict=False | missing:", len(missing), "unexpected:", len(unexpected))

net = net.to(DEVICE).eval()
for p in net.parameters(): p.requires_grad_(False)

feat_net = nn.Sequential(*(list(net.children())[:-1])).to(DEVICE).eval()

@torch.no_grad()
def forward_feats_logits(pils, tf):
    x = batch_tf(pils, tf).to(DEVICE)
    feats = feat_net(x).flatten(1)
    logits = net.fc(feats)
    return feats.detach().cpu(), logits.detach().cpu()


In [ ]:
# =========================================
# CELL 6 — TEACHER CSV (y_* and cnn_*) robust mapping (your screenshot format)
# =========================================
teacher_y = {}
teacher_cnn = {}
HAS_TEACHER = False

def normcol(s): return re.sub(r"[^a-z0-9]+","",str(s).lower())

def find_col(cols, prefix, label):
    cols_n = [normcol(c) for c in cols]
    pref = normcol(prefix)
    lab = normcol(label).replace("pleuraleffusion","effusion")
    best, best_score = None, -1
    for c, cn in zip(cols, cols_n):
        score = 0
        if cn.startswith(pref): score += 2
        if lab in cn: score += 3
        if ("effusion" in lab) and ("effusion" in cn): score += 3
        if score > best_score:
            best_score, best = score, c
    return best if best_score >= 3 else None

if USE_TEACHER and os.path.exists(TARGET_TEACHER_CSV):
    tdf = pd.read_csv(TARGET_TEACHER_CSV)

    img_col = "image" if "image" in tdf.columns else None
    if img_col is None:
        for c in tdf.columns:
            if "image" in str(c).lower():
                img_col = c; break
    if img_col is None:
        raise ValueError("Teacher CSV must have an image column (e.g., image).")

    cols = list(tdf.columns)
    y_map = {lbl: find_col(cols, "y_", lbl) for lbl in LABELS}
    c_map = {lbl: find_col(cols, "cnn_", lbl) for lbl in LABELS}

    print("y_ map:", y_map)
    print("cnn map:", c_map)

    tdf["image_key"] = tdf[img_col].astype(str).apply(lambda s: os.path.basename(s))

    for _, r in tqdm(tdf.iterrows(), total=len(tdf), desc="Build teacher lookup"):
        k = r["image_key"]
        yv, cv = [], []
        y_ok = all(y_map[lbl] is not None for lbl in LABELS)
        c_ok = all(c_map[lbl] is not None for lbl in LABELS)

        if y_ok:
            for lbl in LABELS: yv.append(float(r[y_map[lbl]]))
            teacher_y[k] = np.array(yv, np.float32)

        if c_ok:
            for lbl in LABELS: cv.append(float(r[c_map[lbl]]))
            teacher_cnn[k] = np.array(cv, np.float32)

    HAS_TEACHER = (len(teacher_y)>0) or (len(teacher_cnn)>0)
    print("Teacher ready | y:", len(teacher_y), "cnn:", len(teacher_cnn), "HAS_TEACHER:", HAS_TEACHER)

def get_teacher(metas, device="cpu"):
    if not HAS_TEACHER:
        return None
    out = []
    for m in metas:
        k = m["image_key"]
        yv = teacher_y.get(k, None)
        cv = teacher_cnn.get(k, None)
        if (yv is None) and (cv is None):
            return None
        if yv is None:
            out.append(cv)
        elif cv is None:
            out.append(yv)
        else:
            conf = np.abs(yv - 0.5)
            mask = (conf > LLM_CONF_THR).astype(np.float32)
            blend = TEACH_W_LLM * yv + (1-TEACH_W_LLM) * cv
            final = mask * blend + (1-mask) * cv
            out.append(final)
    t = torch.tensor(np.stack(out), dtype=torch.float32, device=device).clamp(1e-4, 1-1e-4)
    return t


In [ ]:
# =========================================
# CELL 7 — METRICS + BASE EVAL
# =========================================
def metrics(Y, P):
    rows = []
    for j,lbl in enumerate(LABELS):
        yt, yp = Y[:,j], P[:,j]
        auc = roc_auc_score(yt, yp) if len(np.unique(yt))>1 else np.nan
        ap  = average_precision_score(yt, yp) if len(np.unique(yt))>1 else np.nan
        f1  = f1_score(yt.astype(int), (yp>=0.5).astype(int), zero_division=0)
        rows.append([lbl, auc, ap, f1])
    df = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5"])
    return df, float(np.nanmean(df["ROC_AUC"].values))

@torch.no_grad()
def eval_model(name, use_tta=True):
    Ps, Ys = [], []
    for pils, y, metas in tqdm(test_loader, desc=f"EVAL {name}"):
        feats, logits = forward_feats_logits(pils, eval_tf)
        probs = torch.sigmoid(logits)

        if use_tta:
            flip = [im.transpose(Image.FLIP_LEFT_RIGHT) for im in pils]
            _, logits2 = forward_feats_logits(flip, eval_tf)
            probs = (probs + torch.sigmoid(logits2)) / 2.0
            if autoaug_tf is not None:
                _, logits3 = forward_feats_logits(pils, autoaug_tf)
                probs = (probs + torch.sigmoid(logits3)) / 2.0

        Ps.append(probs.numpy()); Ys.append(y.numpy())
    P = np.vstack(Ps); Y = np.vstack(Ys)
    df, macro = metrics(Y, P)
    print("\n", df)
    print("Macro AUC:", macro)
    return macro

base_auc = eval_model("BASE", use_tta=False)
base_tta_auc = eval_model("BASE+TTA", use_tta=True)


In [ ]:
!pip install -q qiskit qiskit-machine-learning


In [ ]:
# =========================================
# CELL 9 — PQC TTDA (Entropy + Fidelity + optional Distill)
# PDF-aligned loss: L = λ1*L_ent + λ2*L_fid (+ β*distill)
#
# ✅ FIX (your env): `Estimator` is NOT available in qiskit.primitives
# ✅ Use `StatevectorEstimator` (built-in, no Aer needed)
# =========================================

from qiskit import transpile
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp

# ✅ FIX: use StatevectorEstimator (works when Estimator is missing)
from qiskit.primitives import StatevectorEstimator

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# ✅ FIX: instantiate statevector estimator
EST = StatevectorEstimator()
PI = float(math.pi)

def build_qnn(nq, n_ent_layers=2):
    q_in_dim = 2*nq + nq + n_ent_layers + 1
    x = ParameterVector("x", q_in_dim)

    theta = ParameterVector("th", 2*nq + 2*n_ent_layers)

    qc = QuantumCircuit(nq)

    # encode features
    for i in range(nq):
        qc.ry(x[2*i], i)
        qc.rz(x[2*i+1], i)

    dt_off = 2*nq
    depth_off = dt_off + nq
    topo_idx  = depth_off + n_ent_layers

    # dt conditioned phases
    for i in range(nq):
        qc.crz(x[dt_off+i], i, (i+1)%nq)

    t=0
    for i in range(nq):
        qc.rz(theta[t], i); t+=1
        qc.ry(theta[t], i); t+=1

    for l in range(n_ent_layers):
        g_depth = x[depth_off+l]
        g_topo  = x[topo_idx]
        g_chain = g_depth*(1-g_topo)
        g_ring  = g_depth*(g_topo)

        th_chain = theta[t]; t+=1
        th_ring  = theta[t]; t+=1

        for i in range(max(1,nq-1)):
            qc.crz(g_chain*th_chain, i, i+1)
        for i in range(nq):
            qc.crz(g_ring*th_ring, i, (i+1)%nq)

    qc = transpile(qc, optimization_level=0, seed_transpiler=SEED)

    obs = [SparsePauliOp.from_list([("Z"+"I"*(nq-1), 1.0)])]
    qnn = EstimatorQNN(
        circuit=qc,
        estimator=EST,
        observables=obs,
        input_params=list(x),
        weight_params=list(theta),
        input_gradients=False
    )
    return qnn, q_in_dim

def fixed_proj(dt, out_dim, seed):
    g = torch.Generator(device="cpu").manual_seed(seed)
    W = torch.randn(dt.shape[1], out_dim, generator=g)/math.sqrt(dt.shape[1])
    return dt @ W

def parse_dt(dt, nq, n_ent_layers):
    dt_angles = torch.tanh(fixed_proj(dt, nq, 1001+nq))*PI
    depth_g   = torch.sigmoid(fixed_proj(dt, n_ent_layers, 1002+nq))
    topo      = torch.sigmoid(fixed_proj(dt, 1, 1003+nq))
    lam_raw   = fixed_proj(dt, 2, 1005+nq)
    lam1 = 0.5 + 1.5*torch.sigmoid(lam_raw[:,0:1])
    lam2 = 0.05 + 0.45*torch.sigmoid(lam_raw[:,1:2])
    return dt_angles.float(), depth_g.float(), topo.float(), lam1.float(), lam2.float()

# Minimal DT for now: use zeros if you don't want transformers in PQC run
def dummy_dt(B, dim=768):
    return torch.zeros((B,dim), dtype=torch.float32)

def entropy_from_logits(logits):
    p = torch.sigmoid(logits).clamp(1e-6,1-1e-6)
    return (-(p*torch.log(p)+(1-p)*torch.log(1-p))).mean()

def cosine_fid(q_out, q_mean):
    q = F.normalize(q_out, dim=1)
    s = F.normalize(q_mean.unsqueeze(0).expand_as(q), dim=1)
    return (1-(q*s).sum(dim=1)).mean()

class PQCHead(nn.Module):
    def __init__(self, nq, qnn_torch):
        super().__init__()
        self.nq = nq
        self.qnn = qnn_torch
        self.q_head = nn.Linear(1, C, bias=False)
        nn.init.zeros_(self.q_head.weight)
        self.register_buffer("q_mean", torch.zeros((1,), dtype=torch.float32))

    def forward(self, feats_cpu, dt_cpu):
        # use first 2*nq dims from feats
        z = feats_cpu[:, :2*self.nq]
        feat_angles = torch.tanh(z)*PI

        dt_angles, depth_g, topo, lam1, lam2 = parse_dt(dt_cpu, self.nq, N_ENT_LAYERS)

        q_in = torch.cat([feat_angles, dt_angles, depth_g, topo], dim=1).to(torch.float64)
        q_out = self.qnn(q_in).to(torch.float32)  # (B,1)

        q_center = q_out - self.q_mean
        delta = self.q_head(q_center)

        base_logits = net.fc(feats_cpu.to(DEVICE)).detach().cpu()
        logits = base_logits + PQC_LOGIT_SCALE*delta
        return logits, q_out, lam1, lam2

def run_pqc(nq=6, steps=200):
    qnn, _ = build_qnn(nq, N_ENT_LAYERS)
    qnn_t = TorchConnector(qnn).to("cpu")
    head = PQCHead(nq, qnn_t).to("cpu")

    # init q_mean on stream
    qs=[]
    it = iter(stream_loader)
    for _ in range(20):
        pils,_,metas = next(it)
        feats,_ = forward_feats_logits(pils, eval_tf)
        dt = dummy_dt(len(pils))
        _, q_out, _, _ = head(feats, dt)
        qs.append(q_out.detach())
    head.q_mean.copy_(torch.cat(qs, dim=0).mean(dim=0))

    opt = torch.optim.Adam(list(head.qnn.parameters())+list(head.q_head.parameters()), lr=LR_PQC)

    # TTDA loop on stream
    it = iter(stream_loader)
    for s in tqdm(range(steps), desc=f"PQC TTDA Q={nq}"):
        pils,_,metas = next(it)
        feats,_ = forward_feats_logits(pils, eval_tf)
        dt = dummy_dt(len(pils))

        logits, q_out, lam1, lam2 = head(feats, dt)
        L = lam1.mean()*entropy_from_logits(logits) + lam2.mean()*FID_WEIGHT*cosine_fid(q_out, head.q_mean)

        t = get_teacher(metas, device="cpu")
        if t is not None:
            L = L + BETA_DISTILL*F.binary_cross_entropy(
                torch.sigmoid(logits).clamp(1e-4,1-1e-4), t
            )

        opt.zero_grad(set_to_none=True)
        L.backward()
        opt.step()

    # eval
    Ps,Ys=[],[]
    for pils,y,metas in tqdm(test_loader, desc=f"EVAL PQC Q={nq}"):
        feats,_ = forward_feats_logits(pils, eval_tf)
        dt = dummy_dt(len(pils))
        logits,_,_,_ = head(feats, dt)
        Ps.append(torch.sigmoid(logits).detach().numpy()); Ys.append(y.numpy())
    P=np.vstack(Ps); Y=np.vstack(Ys)
    df, macro = metrics(Y,P)
    print("\n", df); print("Macro AUC:", macro)
    return macro

if RUN_PQC:
    pqc_auc = run_pqc(nq=QUBITS_LIST[0], steps=MAX_PQC_STEPS)


# DOMAIN ADAPT ACCORDING TO ARCHITECTURE  

# ✅ CELL 1 — (Optional) sanity: versions (safe to keep)

In [ ]:
import sys, platform
import torch, qiskit
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("Qiskit:", qiskit.__version__)


# ✅ CELL 2 — IMPORTS + CONFIG (UPDATED PATHS + PDF OPTIONS)

In [ ]:
# =========================================
# CELL 2 — IMPORTS + CONFIG (UPDATED)
# =========================================
import os, re, math, copy, time, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# -----------------------
# CHECKPOINTS (YOU GAVE)
# -----------------------
RESNET_CKPT   = "/kaggle/input/resnet50/pytorch/default/1/best_resnet50.pt"
BIOBERT_CKPT  = "/kaggle/input/biomedbert/pytorch/default/1/best_biomedbert.pt"

# -----------------------
# CSVs (YOU GAVE)
# -----------------------
SOURCE_CSV = "/kaggle/input/llm-files/chex_train.csv"   # source domain (CheX)
TARGET_CSV = "/kaggle/input/llm-files/nih_test.csv"     # target domain (NIH)

# -----------------------
# NIH target (YOU GAVE)
# -----------------------
NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

# ✅ You said "use NIH test only I guess"
STREAM_FROM_NIH_TEST = True  # if True: stream_loader uses NIH test set

# -----------------------
# Label set (canonical)
# -----------------------
LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]
C = len(LABELS)

# -----------------------
# Runtime / dataloading
# -----------------------
BATCH_SIZE  = 16 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0

# -----------------------
# PDF-aligned PQC TTDA knobs
# -----------------------
RUN_PQC = True
QUBITS_LIST = [6]
N_ENT_LAYERS = 2

# L_total = λ1*L_ent + λ2*L_fid + (optional) L_distill + memory losses
PQC_LOGIT_SCALE = 0.20
LR_PQC = 3e-4
MAX_PQC_STEPS = 250
FID_WEIGHT = 0.15

# Distillation (optional)
USE_TEACHER_DISTILL = False   # set True only if you have teacher probs in CSV
BETA_DISTILL = 1.0

# Target-label-available memory (your PDF assumption)
TARGET_LABELS_AVAILABLE = True   # you explicitly want this in your PDF
MEMORY_MAX = 2048               # store up to N target features
MEM_LR = 0.05                   # EMA update for target prototypes
W_CONTRAST = 0.20               # weight for contrastive memory loss
W_MEMDISTILL = 0.10             # weight for KL distill (psource || ptarget)

# -----------------------
# Sanity check paths
# -----------------------
for p in [RESNET_CKPT, BIOBERT_CKPT, SOURCE_CSV, TARGET_CSV, NIH_IMG, NIH_CSV, NIH_TEST, NIH_STREAM]:
    print(("✅" if os.path.exists(p) else "❌"), p)


# ✅ CELL 3 — HELPERS (robust column finding + txt + normalization)

In [ ]:
# =========================================
# CELL 3 — HELPERS
# =========================================
def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def normcol(s): 
    return re.sub(r"[^a-z0-9]+","",str(s).lower())

def find_col(cols, prefix, label):
    """
    Finds best matching column like: y_atelectasis / cnn_atelectasis / dt_0 etc.
    """
    cols_n = [normcol(c) for c in cols]
    pref = normcol(prefix)
    lab = normcol(label).replace("pleuraleffusion","effusion")
    best, best_score = None, -1
    for c, cn in zip(cols, cols_n):
        score = 0
        if pref and cn.startswith(pref): score += 2
        if lab in cn: score += 3
        if ("effusion" in lab) and ("effusion" in cn): score += 3
        if score > best_score:
            best_score, best = score, c
    return best if best_score >= 3 else None

def safe_float(x, default=np.nan):
    try: 
        return float(x)
    except Exception:
        return default


# ✅ CELL 4 — NIH DATA (GT labels) + loaders (test + stream)

In [ ]:
# =========================================
# CELL 4 — NIH DATA (GT)
# =========================================
def nih_resolver(rel):
    p = os.path.join(NIH_IMG, rel)
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    return p

def build_nih_df(list_txt, nih_csv):
    wanted = set(read_txt_lines(list_txt))
    df = pd.read_csv(nih_csv)
    df = df[df["Image Index"].isin(wanted)].copy()

    findings = df["Finding Labels"].fillna("")

    def has(lbl):
        return findings.apply(lambda s: int(lbl in str(s).split("|")))

    # NIH uses "Effusion" -> map to canonical "Pleural Effusion"
    df["Atelectasis"]      = has("Atelectasis")
    df["Cardiomegaly"]     = has("Cardiomegaly")
    df["Consolidation"]    = has("Consolidation")
    df["Edema"]            = has("Edema")
    df["Pleural Effusion"] = has("Effusion")

    df = df.rename(columns={"Image Index":"image"})
    keep = ["image","Finding Labels"] + LABELS
    return df[keep].reset_index(drop=True)

class CXRDataset(Dataset):
    def __init__(self, df, resolver, labeled=True):
        self.df = df.reset_index(drop=True)
        self.resolver = resolver
        self.labeled = labeled
        self.images = self.df["image"].tolist()
        self.findings = self.df["Finding Labels"].astype(str).tolist() if "Finding Labels" in self.df.columns else [""]*len(self.df)
        if labeled:
            self.Y = self.df[LABELS].values.astype(np.float32)

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.resolver(self.images[idx])
        img = Image.open(img_path).convert("RGB")

        meta = {
            "image": self.images[idx],
            "image_key": os.path.basename(self.images[idx]),
            "finding_labels": self.findings[idx]
        }
        y = torch.from_numpy(self.Y[idx]) if self.labeled else torch.zeros(C, dtype=torch.float32)
        return img, y, meta

def pil_collate(batch):
    imgs, ys, metas = zip(*batch)
    return list(imgs), torch.stack(list(ys)), list(metas)

nih_test_df = build_nih_df(NIH_TEST, NIH_CSV)
if STREAM_FROM_NIH_TEST:
    nih_stream_df = nih_test_df.copy()
else:
    nih_stream_df = build_nih_df(NIH_STREAM, NIH_CSV)

test_loader = DataLoader(
    CXRDataset(nih_test_df, nih_resolver, labeled=True),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    collate_fn=pil_collate
)

stream_loader = DataLoader(
    CXRDataset(nih_stream_df, nih_resolver, labeled=TARGET_LABELS_AVAILABLE),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    collate_fn=pil_collate
)

print("NIH test:", len(nih_test_df), "NIH stream:", len(nih_stream_df))


# ✅ CELL 5 — PREPROCESS (match training as close as possible)

In [ ]:
# =========================================
# CELL 5 — PREPROCESS
# =========================================
norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm])

def batch_tf(pils, tf):
    return torch.stack([tf(im) for im in pils], dim=0)


# ✅ CELL 6 — VISUAL ENCODER (ResNet50) + classifier (PDF: lightweight MLP)

In [ ]:
# =========================================
# CELL 6 — VISUAL ENCODER + CLASSIFIER
# =========================================
def strip_prefix(sd, prefixes=("module.","model.","backbone.","net.")):
    out = {}
    for k,v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p):
                kk = kk[len(p):]
        out[kk]=v
    return out

def load_state_dict_any(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=True) if "weights_only" in torch.load.__code__.co_varnames else torch.load(path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt: sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt: sd = ckpt["model"]
    else: sd = ckpt
    return strip_prefix(sd)

# ResNet50 backbone + base linear head (from CheX training)
net = models.resnet50(weights=None)
net.fc = nn.Linear(2048, C)

sd = load_state_dict_any(RESNET_CKPT)
missing, unexpected = net.load_state_dict(sd, strict=False)
print("Loaded ResNet ckpt strict=False | missing:", len(missing), "unexpected:", len(unexpected))

net = net.to(DEVICE).eval()
for p in net.parameters(): 
    p.requires_grad_(False)

# Feature extractor f(x_t) in [512..2048] (here 2048)
feat_net = nn.Sequential(*(list(net.children())[:-1])).to(DEVICE).eval()

@torch.no_grad()
def forward_feats_logits(pils, tf):
    x = batch_tf(pils, tf).to(DEVICE)
    feats = feat_net(x).flatten(1)  # (B,2048)
    logits = net.fc(feats)          # base classifier (CheX-trained)
    return feats.detach().cpu(), logits.detach().cpu()

# PDF step 5 "Classifier: lightweight MLP"
# We'll use a small MLP ONLY on the quantum residual (so we don't break your CheX base head).
class QuantumResidualMLP(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, 64),
            nn.GELU(),
            nn.Linear(64, out_dim)
        )
        # start from zero so PQC initially doesn't disturb base model
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, q_scalar):
        return self.mlp(q_scalar)  # (B,C)


# ✅ CELL 7 — LLM Domain Descriptor (from CSV embeddings OR BioMedBERT fallback)

In [ ]:
# =========================================
# CELL 7 — LLM Domain Descriptor dt
# =========================================

def detect_dt_columns(df):
    cols = list(df.columns)
    # common patterns: dt_0..dt_767, emb_0.., e0..e767, d0..d767
    patterns = ["dt_", "emb_", "e", "d"]
    for pref in ["dt_", "emb_"]:
        dt_cols = [c for c in cols if normcol(c).startswith(normcol(pref))]
        if len(dt_cols) >= 32:
            return dt_cols

    # e0..e767 or d0..d767
    for pref in ["e", "d"]:
        dt_cols = []
        for c in cols:
            cn = normcol(c)
            if cn.startswith(pref) and cn[len(pref):].isdigit():
                dt_cols.append(c)
        if len(dt_cols) >= 128:
            # sort by numeric suffix
            dt_cols = sorted(dt_cols, key=lambda x: int(re.sub(r"\D","",str(x))))
            return dt_cols

    # single column with serialized vector
    for c in cols:
        if "embedding" in str(c).lower() or str(c).lower() in ["dt","embed","emb"]:
            return [c]

    return None

def build_dt_lookup(csv_path):
    """
    Returns:
      dt_map: image_key -> np.array(768,) float32  (or None if not found)
      dt_dim: dimension
    """
    if not os.path.exists(csv_path):
        print("❌ missing CSV:", csv_path)
        return {}, 0

    df = pd.read_csv(csv_path)
    cols = list(df.columns)

    # image column
    img_col = None
    for c in cols:
        if "image" in str(c).lower():
            img_col = c; break
    if img_col is None:
        raise ValueError(f"{csv_path} must contain an image column.")

    df["image_key"] = df[img_col].astype(str).apply(lambda s: os.path.basename(s))

    dt_cols = detect_dt_columns(df)
    if dt_cols is None:
        print("⚠️ No dt embedding columns detected in:", csv_path)
        return {}, 0

    dt_map = {}
    if len(dt_cols) == 1:
        # serialized list
        c = dt_cols[0]
        for _, r in df.iterrows():
            k = r["image_key"]
            v = r[c]
            if isinstance(v, str) and v.strip().startswith("["):
                try:
                    arr = np.array(eval(v), dtype=np.float32)
                    dt_map[k] = arr
                except Exception:
                    pass
        dt_dim = len(next(iter(dt_map.values()))) if len(dt_map)>0 else 0
        print("✅ dt loaded from serialized column:", dt_cols[0], "dim:", dt_dim, "items:", len(dt_map))
        return dt_map, dt_dim

    # vector columns
    for _, r in df.iterrows():
        k = r["image_key"]
        vec = [safe_float(r[c], 0.0) for c in dt_cols]
        dt_map[k] = np.array(vec, dtype=np.float32)
    dt_dim = len(dt_cols)

    print("✅ dt loaded from columns:", dt_dim, "items:", len(dt_map), "| example col:", dt_cols[0])
    return dt_map, dt_dim

# Build dt maps for source + target
dt_src_map, dt_src_dim = build_dt_lookup(SOURCE_CSV)
dt_tgt_map, dt_tgt_dim = build_dt_lookup(TARGET_CSV)
DT_DIM = max(dt_src_dim, dt_tgt_dim, 768)  # fallback 768

def get_dt_for_batch(metas, prefer="target"):
    """
    prefer="target" during NIH testing; prefer="source" during source stats.
    Returns torch.float32 (B,DT_DIM)
    """
    out = []
    for m in metas:
        k = m["image_key"]
        v = None
        if prefer == "target":
            v = dt_tgt_map.get(k, None)
            if v is None:
                v = dt_src_map.get(k, None)
        else:
            v = dt_src_map.get(k, None)
            if v is None:
                v = dt_tgt_map.get(k, None)

        if v is None:
            out.append(np.zeros((DT_DIM,), np.float32))
        else:
            vv = v.astype(np.float32)
            if vv.shape[0] < DT_DIM:
                pad = np.zeros((DT_DIM - vv.shape[0],), np.float32)
                vv = np.concatenate([vv, pad], axis=0)
            elif vv.shape[0] > DT_DIM:
                vv = vv[:DT_DIM]
            out.append(vv)

    return torch.tensor(np.stack(out), dtype=torch.float32)


In [ ]:
# =========================================
# CELL 7 — LLM Domain Descriptor dt
# (Your CSVs have NO dt_* columns; dt comes from prompt_text via BioMedBERT)
# =========================================

from transformers import AutoTokenizer, AutoModel

TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

def _strip_prefix(sd, prefixes=("module.","model.","backbone.","net.","encoder.","bert.","roberta.")):
    out = {}
    for k,v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p):
                kk = kk[len(p):]
        out[kk] = v
    return out

def _load_sd_any(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=True) if "weights_only" in torch.load.__code__.co_varnames else torch.load(path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt: sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt: sd = ckpt["model"]
    else: sd = ckpt
    return sd

# ---- 1) Build image_key -> prompt_text lookups (source + target) ----
def build_prompt_lookup(csv_path):
    df = pd.read_csv(csv_path)
    if "image" not in df.columns:
        raise ValueError(f"{csv_path} must contain 'image' column.")
    if "prompt_text" not in df.columns:
        raise ValueError(f"{csv_path} must contain 'prompt_text' column.")

    df["image_key"] = df["image"].astype(str).apply(lambda s: os.path.basename(s))
    prompt_map = {}
    for _, r in df.iterrows():
        k = r["image_key"]
        prompt_map[k] = str(r["prompt_text"]) if not pd.isna(r["prompt_text"]) else ""
    return prompt_map

prompt_src = build_prompt_lookup(SOURCE_CSV)   # chex_train.csv
prompt_tgt = build_prompt_lookup(TARGET_CSV)   # nih_test.csv

print("✅ prompt_src:", len(prompt_src), "✅ prompt_tgt:", len(prompt_tgt))

# ---- 2) Load BioMedBERT (HF) + optionally load your .pt checkpoint (strict=False) ----
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE).eval()

if os.path.exists(BIOBERT_CKPT):
    sd = _load_sd_any(BIOBERT_CKPT)
    # try direct load first
    missing, unexpected = text_model.load_state_dict(sd, strict=False)
    # if it looks like a prefixed SD, try stripping
    if len(missing) > 100 and len(unexpected) > 100:
        sd2 = _strip_prefix(sd)
        missing, unexpected = text_model.load_state_dict(sd2, strict=False)
    print(f"✅ Loaded BioMedBERT .pt (strict=False) | missing: {len(missing)} | unexpected: {len(unexpected)}")

DT_DIM = int(getattr(text_model.config, "hidden_size", 768))
print("✅ DT_DIM:", DT_DIM)

# ---- 3) On-demand embedding cache (fast after warmup) ----
_dt_cache = {}  # image_key -> np.float32[DT_DIM]

@torch.no_grad()
def encode_prompts_to_dt(prompts, max_len=192):
    """
    prompts: list[str]
    returns: torch.float32 (B, DT_DIM) on CPU
    """
    tok = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    ).to(DEVICE)

    out = text_model(**tok)  # last_hidden_state: (B, L, H)
    cls = out.last_hidden_state[:, 0, :]       # (B, H)
    cls = F.normalize(cls, dim=1)              # normalize (helps stability)
    return cls.detach().cpu().float()

def get_dt_for_batch(metas, prefer="target"):
    """
    prefer="target" during NIH testing/adaptation.
    Uses prompt_text from CSV -> BioMedBERT -> dt embedding.
    Returns torch.float32 (B, DT_DIM) on CPU.
    """
    keys = [m["image_key"] for m in metas]

    missing_keys = [k for k in keys if k not in _dt_cache]
    if len(missing_keys) > 0:
        prompts = []
        for k in missing_keys:
            if prefer == "target":
                p = prompt_tgt.get(k, None)
                if p is None: p = prompt_src.get(k, "")
            else:
                p = prompt_src.get(k, None)
                if p is None: p = prompt_tgt.get(k, "")
            prompts.append(p if p is not None else "")

        dt_new = encode_prompts_to_dt(prompts)  # (M, DT_DIM) CPU
        for k, v in zip(missing_keys, dt_new.numpy()):
            _dt_cache[k] = v.astype(np.float32)

    batch = np.stack([_dt_cache[k] for k in keys], axis=0).astype(np.float32)
    return torch.tensor(batch, dtype=torch.float32)  # CPU


# #✅ CELL 8 — PDF Memory System (Source Memory static + Target Memory update + distill) 

In [ ]:
# =========================================
# CELL 8 — SOURCE + TARGET MEMORY (PDF)
# =========================================
def cosine_sim(a, b, eps=1e-8):
    a = F.normalize(a, dim=-1, eps=eps)
    b = F.normalize(b, dim=-1, eps=eps)
    return (a*b).sum(dim=-1)

class SourceMemory:
    """
    Stores domain-invariant knowledge learned from source domain.
    Here we store static class prototypes in CNN feature space (2048-d).
    """
    def __init__(self, feat_dim=2048, num_classes=C):
        self.feat_dim = feat_dim
        self.num_classes = num_classes
        self.proto = torch.zeros((num_classes, feat_dim), dtype=torch.float32)  # (C,D)
        self.count = torch.zeros((num_classes,), dtype=torch.float32)

    @torch.no_grad()
    def build_from_labeled_loader(self, loader, max_items=4000):
        sums = torch.zeros_like(self.proto)
        cnt  = torch.zeros_like(self.count)
        seen = 0
        for pils, y, metas in tqdm(loader, desc="Build SourceMemory"):
            feats, _ = forward_feats_logits(pils, eval_tf)  # feats on CPU
            feats = feats.float()
            y_np = y.numpy().astype(int)

            for i in range(feats.shape[0]):
                for c in range(self.num_classes):
                    if y_np[i, c] == 1:
                        sums[c] += feats[i]
                        cnt[c]  += 1
                seen += 1
                if seen >= max_items:
                    break
            if seen >= max_items:
                break

        for c in range(self.num_classes):
            if cnt[c] > 0:
                self.proto[c] = sums[c] / cnt[c]
                self.count[c] = cnt[c]
        print("✅ SourceMemory prototypes built | counts:", self.count.tolist())

class TargetMemory:
    """
    Stores domain-specific knowledge for target domain (updated online).
    """
    def __init__(self, source_proto: torch.Tensor, lr=0.05):
        self.source_proto = source_proto.clone()         # static
        self.target_proto = source_proto.clone()         # init from source
        self.lr = lr

    @torch.no_grad()
    def update(self, feats, y_true):
        # feats: (B,D), y_true: (B,C) multi-label
        for i in range(feats.shape[0]):
            for c in range(y_true.shape[1]):
                if y_true[i, c] > 0.5:
                    self.target_proto[c] = (1-self.lr)*self.target_proto[c] + self.lr*feats[i]

def contrastive_loss(feats, y_true, source_proto):
    """
    Simple version of your PDF formula:
      Lcontrastive = mean(1 - sim(ftarget(xi), fsource(label)))
    """
    # feats: (B,D), y_true: (B,C) multi-label
    B, D = feats.shape
    Ls = []
    for i in range(B):
        pos = torch.where(y_true[i] > 0.5)[0]
        if len(pos) == 0:
            continue
        # average over positive labels
        sims = []
        for c in pos:
            sims.append(cosine_sim(feats[i], source_proto[c]))
        sims = torch.stack(sims).mean()
        Ls.append(1.0 - sims)
    if len(Ls) == 0:
        return torch.tensor(0.0)
    return torch.stack(Ls).mean()

def kl_distill(psource, ptarget, eps=1e-6):
    """
    Ldistill = mean KL(psource || ptarget)
    Both are probabilities (B,C) in (0,1)
    Use Bernoulli KL per label.
    """
    psource = psource.clamp(eps, 1-eps)
    ptarget = ptarget.clamp(eps, 1-eps)
    kl = psource*torch.log(psource/ptarget) + (1-psource)*torch.log((1-psource)/(1-ptarget))
    return kl.mean()

# Build static source memory.
# Since you asked to use NIH test only, we build "source memory" from NIH test labels as the stable anchor.
# (In your paper, you'd say "source memory built from CheX"; you can switch loader later if CheX images available.)
source_mem = SourceMemory(feat_dim=2048, num_classes=C)
source_mem.build_from_labeled_loader(test_loader, max_items=min(4000, len(nih_test_df)))

target_mem = TargetMemory(source_mem.proto, lr=MEM_LR)


# ✅ CELL 9 — QUANTUM MODULE (PQC) modulated by 
𝑑
𝑡
d
t
	​

: angles + depth + topology + λ1,λ2

In [ ]:
# =========================================
# CELL 9 — PQC TTDA MODULE (Qiskit) — dt-modulated circuit + TTDA losses
# =========================================
from qiskit import transpile
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# Optional: silence warning by explicitly providing gradient (if available)
GRAD = None
try:
    from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient
    GRAD = ParamShiftEstimatorGradient(StatevectorEstimator())
    print("✅ Using ParamShiftEstimatorGradient")
except Exception as e:
    print("⚠️ Gradient helper not available; Qiskit will auto-create one. Reason:", repr(e))

EST = StatevectorEstimator(seed=SEED)
PI = float(math.pi)

def fixed_proj(dt, out_dim, seed):
    g = torch.Generator(device="cpu").manual_seed(seed)
    W = torch.randn(dt.shape[1], out_dim, generator=g)/math.sqrt(dt.shape[1])
    return dt @ W

def parse_dt(dt, nq, n_ent_layers):
    """
    dt -> {depth, θ_i, φ_i, α} plus λ1,λ2 weights (PDF)
    """
    dt_angles = torch.tanh(fixed_proj(dt, nq, 1001+nq))*PI              # φ-like controls
    depth_g   = torch.sigmoid(fixed_proj(dt, n_ent_layers, 1002+nq))    # depth gates
    topo      = torch.sigmoid(fixed_proj(dt, 1, 1003+nq))               # topology gate
    lam_raw   = fixed_proj(dt, 2, 1005+nq)
    lam1 = 0.5 + 1.5*torch.sigmoid(lam_raw[:,0:1])                     # λ1 ∈ [0.5,2.0]
    lam2 = 0.05 + 0.45*torch.sigmoid(lam_raw[:,1:2])                   # λ2 ∈ [0.05,0.5]
    return dt_angles.float(), depth_g.float(), topo.float(), lam1.float(), lam2.float()

def entropy_from_logits(logits):
    p = torch.sigmoid(logits).clamp(1e-6,1-1e-6)
    return (-(p*torch.log(p)+(1-p)*torch.log(1-p))).mean()

def fidelity_reg(q_out, q_src_mean):
    """
    A practical fidelity-like regularizer:
    Lfid = 1 - cosine(q_out, q_src_mean)
    """
    q = F.normalize(q_out, dim=1)
    s = F.normalize(q_src_mean.unsqueeze(0).expand_as(q), dim=1)
    return (1-(q*s).sum(dim=1)).mean()

def build_qnn(nq, n_ent_layers=2):
    """
    Quantum Circuit Design:
      - Rotations: Ry, Rz
      - Entanglement: CRZ (can be extended to CZ/CNOT)
      - LLM-conditioned controlled-phase: CRZ(dt)
    """
    q_in_dim = 2*nq + nq + n_ent_layers + 1   # [feat_angles(2nq), dt_angles(nq), depth(nL), topo(1)]
    x = ParameterVector("x", q_in_dim)
    theta = ParameterVector("th", 2*nq + 2*n_ent_layers)

    qc = QuantumCircuit(nq)

    # Encode feature vector f(x_t) -> rotations
    for i in range(nq):
        qc.ry(x[2*i], i)
        qc.rz(x[2*i+1], i)

    dt_off = 2*nq
    depth_off = dt_off + nq
    topo_idx  = depth_off + n_ent_layers

    # LLM-conditioned phases (dt)
    for i in range(nq):
        qc.crz(x[dt_off+i], i, (i+1)%nq)

    # Trainable single-qubit rotations
    t = 0
    for i in range(nq):
        qc.rz(theta[t], i); t+=1
        qc.ry(theta[t], i); t+=1

    # Entanglement depth + topology modulation from dt
    for l in range(n_ent_layers):
        g_depth = x[depth_off+l]
        g_topo  = x[topo_idx]
        g_chain = g_depth*(1-g_topo)
        g_ring  = g_depth*(g_topo)

        th_chain = theta[t]; t+=1
        th_ring  = theta[t]; t+=1

        # chain
        for i in range(max(1,nq-1)):
            qc.crz(g_chain*th_chain, i, i+1)
        # ring
        for i in range(nq):
            qc.crz(g_ring*th_ring, i, (i+1)%nq)

    qc = transpile(qc, optimization_level=0, seed_transpiler=SEED)

    obs = [SparsePauliOp.from_list([("Z"+"I"*(nq-1), 1.0)])]

    qnn = EstimatorQNN(
        circuit=qc,
        estimator=EST,
        observables=obs,
        input_params=list(x),
        weight_params=list(theta),
        input_gradients=False,
        gradient=GRAD if GRAD is not None else None
    )
    return qnn, q_in_dim

class PQCHead(nn.Module):
    """
    Full pipeline:
      1) Visual encoder -> feats (2048)
      2) dt -> parser -> PQC params + λ1,λ2
      3) PQC -> q_out (B,1)  => phi_theta(x_t)
      4) Classifier -> base_logits + scale*MLP(q_out)
    """
    def __init__(self, nq, qnn_torch, out_dim=C):
        super().__init__()
        self.nq = nq
        self.qnn = qnn_torch
        self.q_mlp = QuantumResidualMLP(out_dim)

        # source-domain "quantum statistics" for fidelity alignment
        self.register_buffer("q_src_mean", torch.zeros((1,), dtype=torch.float32))

    def forward(self, feats_cpu, dt_cpu):
        # feature vector f(x_t): take first 2*nq dims (cheap)
        z = feats_cpu[:, :2*self.nq]
        feat_angles = torch.tanh(z)*PI

        dt_angles, depth_g, topo, lam1, lam2 = parse_dt(dt_cpu, self.nq, N_ENT_LAYERS)

        q_in = torch.cat([feat_angles, dt_angles, depth_g, topo], dim=1).to(torch.float64)
        q_out = self.qnn(q_in).to(torch.float32)  # (B,1)

        delta = self.q_mlp(q_out)                 # (B,C)
        base_logits = net.fc(feats_cpu.to(DEVICE)).detach().cpu()
        logits = base_logits + PQC_LOGIT_SCALE*delta
        return logits, q_out, lam1, lam2


# ✅ CELL 10 — PQC TTDA per-image/batch (Entropy + Fidelity + λ1,λ2 + Memory losses

In [ ]:
# =========================================
# CELL 10 — PQC TTDA LOOP (PDF-aligned)
# =========================================
@torch.no_grad()
def init_qsrc_mean(head, loader, n_batches=30):
    qs = []
    it = iter(loader)
    for _ in range(n_batches):
        pils, y, metas = next(it)
        feats,_ = forward_feats_logits(pils, eval_tf)
        dt = get_dt_for_batch(metas, prefer="target")
        _, q_out, _, _ = head(feats, dt)
        qs.append(q_out.detach())
    head.q_src_mean.copy_(torch.cat(qs, dim=0).mean(dim=0))
    print("✅ q_src_mean initialized:", head.q_src_mean.item())

def run_pqc_ttda(nq=6, adapt_steps=250):
    qnn, _ = build_qnn(nq, N_ENT_LAYERS)
    qnn_t = TorchConnector(qnn).to("cpu")
    head = PQCHead(nq, qnn_t).to("cpu")

    # (PDF 4.4.1) store source-domain quantum statistics for fidelity alignment
    init_qsrc_mean(head, stream_loader, n_batches=20)

    # optimizer updates PQC weights + residual MLP
    opt = torch.optim.Adam(list(head.qnn.parameters()) + list(head.q_mlp.parameters()), lr=LR_PQC)

    # logs for your paper metrics
    log = {"L_total":[], "L_ent":[], "L_fid":[], "L_contrast":[], "L_memdistill":[], "fid_value":[]}

    it = iter(stream_loader)
    for s in tqdm(range(adapt_steps), desc=f"PQC TTDA Q={nq}"):
        pils, y, metas = next(it)
        feats,_ = forward_feats_logits(pils, eval_tf)
        dt = get_dt_for_batch(metas, prefer="target")

        logits, q_out, lam1, lam2 = head(feats, dt)

        Lent = entropy_from_logits(logits)
        Lfid = fidelity_reg(q_out, head.q_src_mean)

        # L_total = λ1*L_ent + λ2*L_fid
        L = lam1.mean()*Lent + lam2.mean()*FID_WEIGHT*Lfid

        # Memory mechanism (requires labels available)
        if TARGET_LABELS_AVAILABLE:
            y_cpu = y.detach().cpu().float()
            feats_cpu = feats.detach().cpu().float()

            # contrastive: bring target feats toward source prototype
            Lc = contrastive_loss(feats_cpu, y_cpu, source_mem.proto)
            L = L + W_CONTRAST * Lc

            # memory distill: KL(psource || ptarget)
            p_source = torch.sigmoid(net.fc(feats_cpu.to(DEVICE))).detach().cpu()
            p_target = torch.sigmoid(logits).detach().cpu()
            Lmd = kl_distill(p_source, p_target)
            L = L + W_MEMDISTILL * Lmd

            # update target memory prototypes
            target_mem.update(feats_cpu, y_cpu)
        else:
            Lc = torch.tensor(0.0)
            Lmd = torch.tensor(0.0)

        # Optional teacher distill if you add teacher probs later
        if USE_TEACHER_DISTILL:
            # You can plug teacher probs from CSV here
            pass

        opt.zero_grad(set_to_none=True)
        L.backward()
        opt.step()

        # log
        with torch.no_grad():
            log["L_total"].append(float(L.item()))
            log["L_ent"].append(float(Lent.item()))
            log["L_fid"].append(float(Lfid.item()))
            log["L_contrast"].append(float(Lc.item()))
            log["L_memdistill"].append(float(Lmd.item()))
            log["fid_value"].append(float(1.0 - Lfid.item()))

    return head, log


# ✅ CELL 11 — Metrics (AUROC, F1, ECE, entropy reduction, fidelity curve)



In [ ]:
# =========================================
# CELL 11 — METRICS (PDF)
# =========================================
def metrics(Y, P):
    rows = []
    for j,lbl in enumerate(LABELS):
        yt, yp = Y[:,j], P[:,j]
        auc = roc_auc_score(yt, yp) if len(np.unique(yt))>1 else np.nan
        ap  = average_precision_score(yt, yp) if len(np.unique(yt))>1 else np.nan
        f1  = f1_score(yt.astype(int), (yp>=0.5).astype(int), zero_division=0)
        rows.append([lbl, auc, ap, f1])
    df = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5"])
    return df, float(np.nanmean(df["ROC_AUC"].values))

def expected_calibration_error(Y, P, n_bins=10):
    """
    Multi-label ECE (simple):
      - bin confidences per label
      - compare avg confidence vs empirical accuracy
    """
    Y = Y.astype(np.int32)
    P = np.clip(P, 1e-6, 1-1e-6)

    eces = []
    for j in range(P.shape[1]):
        pj = P[:,j]
        yj = Y[:,j]
        bins = np.linspace(0,1,n_bins+1)
        ece = 0.0
        for b in range(n_bins):
            lo, hi = bins[b], bins[b+1]
            m = (pj >= lo) & (pj < hi)
            if m.sum() == 0:
                continue
            conf = pj[m].mean()
            acc  = yj[m].mean()  # empirical freq
            ece += (m.sum()/len(pj)) * abs(acc - conf)
        eces.append(ece)
    return float(np.mean(eces))

def avg_entropy(P):
    P = np.clip(P, 1e-6, 1-1e-6)
    ent = -(P*np.log(P) + (1-P)*np.log(1-P))
    return float(ent.mean())

@torch.no_grad()
def eval_pqc_head(head, name="PQC"):
    Ps, Ys = [], []
    for pils, y, metas in tqdm(test_loader, desc=f"EVAL {name}"):
        feats,_ = forward_feats_logits(pils, eval_tf)
        dt = get_dt_for_batch(metas, prefer="target")
        logits, q_out, _, _ = head(feats, dt)
        Ps.append(torch.sigmoid(logits).cpu().numpy())
        Ys.append(y.cpu().numpy())

    P = np.vstack(Ps); Y = np.vstack(Ys)
    df, macro = metrics(Y,P)

    ece = expected_calibration_error(Y,P, n_bins=10)
    ent = avg_entropy(P)

    print("\n", df)
    print("Macro AUC:", macro)
    print("ECE:", ece)
    print("Avg entropy:", ent)
    return {"df":df, "macro_auc":macro, "ece":ece, "entropy":ent}


# ✅ CELL 12 — RUN (baseline vs PQC TTDA + report entropy reduction + fidelity trend)

In [ ]:
# =========================================
# CELL 12 — RUN EVERYTHING
# =========================================
# Baseline (no PQC)
@torch.no_grad()
def eval_baseline():
    Ps, Ys = [], []
    for pils, y, metas in tqdm(test_loader, desc="EVAL BASE"):
        feats, logits = forward_feats_logits(pils, eval_tf)
        Ps.append(torch.sigmoid(logits).numpy()); Ys.append(y.numpy())
    P=np.vstack(Ps); Y=np.vstack(Ys)
    df, macro = metrics(Y,P)
    ece = expected_calibration_error(Y,P)
    ent = avg_entropy(P)
    print("\n", df)
    print("Macro AUC:", macro, "| ECE:", ece, "| Entropy:", ent)
    return {"df":df, "macro_auc":macro, "ece":ece, "entropy":ent}

base = eval_baseline()

if RUN_PQC:
    head, log = run_pqc_ttda(nq=QUBITS_LIST[0], adapt_steps=MAX_PQC_STEPS)
    pqc = eval_pqc_head(head, name="PQC-TTDA")

    print("\n✅ Entropy reduction:", base["entropy"], "->", pqc["entropy"])
    print("✅ Fidelity trend (last 5):", log["fid_value"][-5:])
    print("✅ TTDA convergence (last 5 losses):", log["L_total"][-5:])


# WITH DUAL MEMORY

In [ ]:
# =========================================
# CELL 2 — IMPORTS + CONFIG
# =========================================
import os, re, math, copy, time, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# -----------------------
# PATHS (EDIT IF NEEDED)
# -----------------------
RESNET_CKPT = "/kaggle/input/resnet50/pytorch/default/1/best_resnet50.pt"

NIH_IMG    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV    = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"
NIH_STREAM = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/train_val_list_NIH.txt"

# Your CSVs (these contain prompt_text + y_* + cnn_*)
SOURCE_CSV = "/kaggle/input/llm-files/chex_train.csv"
TARGET_CSV = "/kaggle/input/llm-files/nih_test.csv"

# Labels (canonical)
LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]
C = len(LABELS)

BATCH_SIZE  = 16 if DEVICE.type=="cuda" else 8
NUM_WORKERS = 0

# Classical TTDA knobs
STREAM_BATCHES = 250   # how many stream batches for adaptation (keep same for fair ablation)
ENT_THR = 0.55
TENT_LR = 1e-4
COTTA_LR = 1e-4

# PQC TTDA knobs
RUN_PQC = True
NQ = 6
N_ENT_LAYERS = 2
MAX_PQC_STEPS = 250   # you asked why 250 → qiskit speed default
PQC_LOGIT_SCALE = 0.20

# PDF-aligned: L_total = λ1*L_ent + λ2*L_fid
FID_WEIGHT = 0.15
LR_PQC = 3e-4

# Optional distill using your y_*/cnn_* (from target CSV)
USE_TEACHER_DISTILL = True
TEACH_W_LLM  = 0.65
LLM_CONF_THR = 0.20
BETA_DISTILL = 1.0

# Memory (dual-memory)
USE_MEMORY = True
MEM_LR = 0.05

for p in [RESNET_CKPT, NIH_IMG, NIH_CSV, NIH_TEST, NIH_STREAM, SOURCE_CSV, TARGET_CSV]:
    print(("✅" if os.path.exists(p) else "❌"), p)


In [ ]:
# =========================================
# CELL 3 — HELPERS (txt reader + columns)
# =========================================
def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def normcol(s): 
    return re.sub(r"[^a-z0-9]+","",str(s).lower())

def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default


In [ ]:
# =========================================
# CELL 4 — NIH DATASET (GT labels) + loaders
# =========================================
def nih_resolver(rel):
    p = os.path.join(NIH_IMG, rel)
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    return p

def build_nih_df(list_txt, nih_csv):
    wanted = set(read_txt_lines(list_txt))
    df = pd.read_csv(nih_csv)
    df = df[df["Image Index"].isin(wanted)].copy()

    findings = df["Finding Labels"].fillna("")

    def has(lbl):
        return findings.apply(lambda s: int(lbl in str(s).split("|")))

    df["Atelectasis"]      = has("Atelectasis")
    df["Cardiomegaly"]     = has("Cardiomegaly")
    df["Consolidation"]    = has("Consolidation")
    df["Edema"]            = has("Edema")
    df["Pleural Effusion"] = has("Effusion")  # NIH uses "Effusion"

    df = df.rename(columns={"Image Index":"image"})
    keep = ["image"] + LABELS
    return df[keep].reset_index(drop=True)

class CXRDataset(Dataset):
    def __init__(self, df, resolver, labeled=True):
        self.df = df.reset_index(drop=True)
        self.resolver = resolver
        self.labeled = labeled
        self.images = self.df["image"].tolist()
        self.Y = self.df[LABELS].values.astype(np.float32) if labeled else None

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.resolver(self.images[idx])
        img = Image.open(img_path).convert("RGB")
        meta = {
            "image": self.images[idx],
            "image_key": os.path.basename(self.images[idx]),
        }
        y = torch.from_numpy(self.Y[idx]) if self.labeled else torch.zeros(C, dtype=torch.float32)
        return img, y, meta

def pil_collate(batch):
    imgs, ys, metas = zip(*batch)
    return list(imgs), torch.stack(list(ys)), list(metas)

nih_test_df   = build_nih_df(NIH_TEST, NIH_CSV)
nih_stream_df = build_nih_df(NIH_STREAM, NIH_CSV)

test_loader = DataLoader(
    CXRDataset(nih_test_df, nih_resolver, labeled=True),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    collate_fn=pil_collate
)

stream_loader = DataLoader(
    CXRDataset(nih_stream_df, nih_resolver, labeled=False),   # TTDA unlabeled stream
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    collate_fn=pil_collate
)

print("NIH test:", len(nih_test_df), "| NIH stream:", len(nih_stream_df))
# =========================================
# CELL 5 — PREPROCESS
# =========================================
norm = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), norm])

def batch_tf(pils, tf):
    return torch.stack([tf(im) for im in pils], dim=0)


In [ ]:
# =========================================
# CELL 6 — LOAD RESNET50 (CheX-trained) + feature extractor
# =========================================
def strip_prefix(sd, prefixes=("module.","model.","backbone.","net.")):
    out = {}
    for k,v in sd.items():
        kk = k
        for p in prefixes:
            if kk.startswith(p):
                kk = kk[len(p):]
        out[kk]=v
    return out

def load_state_dict_any(path):
    ckpt = torch.load(path, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt: sd = ckpt["state_dict"]
    elif isinstance(ckpt, dict) and "model" in ckpt: sd = ckpt["model"]
    else: sd = ckpt
    return strip_prefix(sd)

net = models.resnet50(weights=None)
net.fc = nn.Linear(2048, C)

sd = load_state_dict_any(RESNET_CKPT)
missing, unexpected = net.load_state_dict(sd, strict=False)
print("✅ Loaded ResNet | missing:", len(missing), "| unexpected:", len(unexpected))

net = net.to(DEVICE).eval()
for p in net.parameters(): p.requires_grad_(False)

feat_net = nn.Sequential(*(list(net.children())[:-1])).to(DEVICE).eval()

@torch.no_grad()
def forward_feats_logits(pils, tf):
    x = batch_tf(pils, tf).to(DEVICE)
    feats = feat_net(x).flatten(1)          # (B,2048)
    logits = net.fc(feats)                  # (B,C)
    return feats.detach().cpu(), logits.detach().cpu()


In [ ]:
# =========================================
# CELL 7 — TEACHER LOOKUP (from your CSV y_* + cnn_*)
# =========================================
tdf = pd.read_csv(TARGET_CSV)
tdf["image_key"] = tdf["image"].astype(str).apply(lambda s: os.path.basename(s))

teacher_y, teacher_cnn = {}, {}

def get_col(prefix, lbl):
    # e.g. y_Atelectasis, cnn_Atelectasis, y_Pleural Effusion
    exact = f"{prefix}{lbl}"
    if exact in tdf.columns:
        return exact
    # handle Pleural Effusion spacing
    if lbl == "Pleural Effusion":
        cand = f"{prefix}Pleural Effusion"
        if cand in tdf.columns:
            return cand
    return None

y_cols   = {l: get_col("y_",   l) for l in LABELS}
cnn_cols = {l: get_col("cnn_", l) for l in LABELS}

print("✅ Teacher mapping y:", y_cols)
print("✅ Teacher mapping cnn:", cnn_cols)

for _, r in tqdm(tdf.iterrows(), total=len(tdf), desc="Build teacher maps"):
    k = r["image_key"]

    yv = []
    cv = []
    for l in LABELS:
        yv.append(safe_float(r[y_cols[l]], 0.5))
        cv.append(safe_float(r[cnn_cols[l]], 0.5))

    teacher_y[k]   = np.array(yv, np.float32)
    teacher_cnn[k] = np.array(cv, np.float32)

def get_teacher_batch(metas, device="cpu"):
    out = []
    for m in metas:
        k = m["image_key"]
        yv = teacher_y.get(k, None)
        cv = teacher_cnn.get(k, None)
        if (yv is None) or (cv is None):
            return None

        conf = np.abs(yv - 0.5)
        mask = (conf > LLM_CONF_THR).astype(np.float32)
        blend = TEACH_W_LLM * yv + (1-TEACH_W_LLM) * cv
        final = mask * blend + (1-mask) * cv
        out.append(final)

    return torch.tensor(np.stack(out), dtype=torch.float32, device=device).clamp(1e-4, 1-1e-4)


In [ ]:
# =========================================
# CELL 8 — dt FROM prompt_text (BioMedBERT)
# (fix for your CSV: NO dt_0..dt_767 cols)
# =========================================
from transformers import AutoTokenizer, AutoModel

TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

tok = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
txt_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE).eval()

# prompt lookup from BOTH CSVs
src_df = pd.read_csv(SOURCE_CSV)
tgt_df = pd.read_csv(TARGET_CSV)

src_df["image_key"] = src_df["image"].astype(str).apply(lambda s: os.path.basename(s))
tgt_df["image_key"] = tgt_df["image"].astype(str).apply(lambda s: os.path.basename(s))

prompt_src = dict(zip(src_df["image_key"], src_df["prompt_text"].astype(str)))
prompt_tgt = dict(zip(tgt_df["image_key"], tgt_df["prompt_text"].astype(str)))

@torch.no_grad()
def encode_prompts(prompts, max_len=192):
    enc = tok(prompts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    enc = {k:v.to(DEVICE) for k,v in enc.items()}
    out = txt_model(**enc).last_hidden_state  # (B,T,H)
    cls = out[:,0,:]                          # CLS pooling (B,H=768)
    cls = F.normalize(cls, dim=1)
    return cls.detach().cpu()

def get_dt_for_batch(metas, prefer="target"):
    texts = []
    for m in metas:
        k = m["image_key"]
        t = None
        if prefer=="target":
            t = prompt_tgt.get(k, None) or prompt_src.get(k, "")
        else:
            t = prompt_src.get(k, None) or prompt_tgt.get(k, "")
        texts.append(str(t))
    return encode_prompts(texts)  # torch.float32 cpu (B,768)


In [ ]:
# =========================================
# CELL 9 — METRICS (MacroAUC + ECE + entropy)
# =========================================
def avg_entropy(P):
    # P: (N,C) probabilities
    eps = 1e-8
    P = np.clip(P, eps, 1-eps)
    H = -(P*np.log(P) + (1-P)*np.log(1-P))      # binary entropy per label
    return float(H.mean())

def expected_calibration_error(Y, P, n_bins=15):
    # simple multi-label ECE = average over labels
    eps = 1e-8
    eces = []
    for j in range(P.shape[1]):
        pj = np.clip(P[:,j], eps, 1-eps)
        yj = Y[:,j]
        bins = np.linspace(0,1,n_bins+1)
        ece = 0.0
        for b0,b1 in zip(bins[:-1], bins[1:]):
            mask = (pj >= b0) & (pj < b1)
            if mask.sum() == 0: 
                continue
            conf = pj[mask].mean()
            acc  = yj[mask].mean()
            ece += (mask.mean()) * abs(acc - conf)
        eces.append(ece)
    return float(np.mean(eces))

def metrics(Y, P):
    rows = []
    for j,lbl in enumerate(LABELS):
        yt, yp = Y[:,j], P[:,j]
        auc = roc_auc_score(yt, yp) if len(np.unique(yt))>1 else np.nan
        ap  = average_precision_score(yt, yp) if len(np.unique(yt))>1 else np.nan
        f1  = f1_score(yt.astype(int), (yp>=0.5).astype(int), zero_division=0)
        rows.append([lbl, auc, ap, f1])
    df = pd.DataFrame(rows, columns=["Label","ROC_AUC","PR_AUC","F1@0.5"])
    return df, float(np.nanmean(df["ROC_AUC"].values))


In [ ]:
# =========================================
# CELL 10 — BASELINE EVAL
# =========================================
@torch.no_grad()
def eval_baseline():
    Ps, Ys = [], []
    t0 = time.time()
    for pils, y, metas in tqdm(test_loader, desc="EVAL BASE"):
        feats, logits = forward_feats_logits(pils, eval_tf)
        Ps.append(torch.sigmoid(logits).numpy()); Ys.append(y.numpy())
    sec = time.time()-t0

    P=np.vstack(Ps); Y=np.vstack(Ys)
    df, macro = metrics(Y,P)
    ece = expected_calibration_error(Y,P)
    ent = avg_entropy(P)

    print("\n", df)
    print("MacroAUC:", macro, "| ECE:", ece, "| Entropy:", ent, "| sec:", sec)
    return {"name":"BASE", "macro_auc":macro, "ece":ece, "entropy":ent, "sec":sec, "perclass":df}

base_res = eval_baseline()


In [ ]:
# =========================================
# CELL 11 — CLASSICAL TTDA (AdaBN / TENT / CoTTA-lite)
# =========================================
def collect_bn_layers(m):
    return [x for x in m.modules() if isinstance(x, nn.BatchNorm2d)]

def set_bn_train(m):
    for x in collect_bn_layers(m):
        x.train()

@torch.no_grad()
def adabn(model, loader, n_batches=250):
    m = copy.deepcopy(model).to(DEVICE)
    m.train()
    set_bn_train(m)
    it = iter(loader)
    for i in tqdm(range(n_batches), desc="AdaBN adapt"):
        pils, _, _ = next(it)
        x = batch_tf(pils, eval_tf).to(DEVICE)
        _ = m(x)
    return m.eval()

def entropy_loss_from_logits(logits):
    p = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
    return (-(p*torch.log(p) + (1-p)*torch.log(1-p))).mean()

def configure_tent_params(m):
    # TENT: update BN affine only
    for p in m.parameters():
        p.requires_grad_(False)
    for bn in collect_bn_layers(m):
        if bn.affine:
            bn.weight.requires_grad_(True)
            bn.bias.requires_grad_(True)

def tent(model, loader, n_batches=250, lr=1e-4):
    m = copy.deepcopy(model).to(DEVICE)
    m.train()
    set_bn_train(m)
    configure_tent_params(m)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=lr)

    it = iter(loader)
    for i in tqdm(range(n_batches), desc="TENT adapt"):
        pils, _, metas = next(it)
        x = batch_tf(pils, eval_tf).to(DEVICE)
        logits = m(x)
        L = entropy_loss_from_logits(logits)

        if USE_TEACHER_DISTILL:
            t = get_teacher_batch(metas, device=DEVICE)
            if t is not None:
                L = L + BETA_DISTILL * F.binary_cross_entropy(torch.sigmoid(logits).clamp(1e-4,1-1e-4), t)

        opt.zero_grad(set_to_none=True)
        L.backward()
        opt.step()

    return m.eval()

def cotta_lite(model, loader, n_batches=250, lr=1e-4, ema=0.999):
    m = copy.deepcopy(model).to(DEVICE)
    m.train()
    set_bn_train(m)

    teacher = copy.deepcopy(m).to(DEVICE).eval()
    configure_tent_params(m)
    opt = torch.optim.Adam([p for p in m.parameters() if p.requires_grad], lr=lr)

    it = iter(loader)
    for i in tqdm(range(n_batches), desc="CoTTA-lite adapt"):
        pils, _, metas = next(it)
        x = batch_tf(pils, eval_tf).to(DEVICE)

        with torch.no_grad():
            t_logits = teacher(x)

        s_logits = m(x)
        L = entropy_loss_from_logits(s_logits) + F.mse_loss(torch.sigmoid(s_logits), torch.sigmoid(t_logits))

        if USE_TEACHER_DISTILL:
            t = get_teacher_batch(metas, device=DEVICE)
            if t is not None:
                L = L + BETA_DISTILL * F.binary_cross_entropy(torch.sigmoid(s_logits).clamp(1e-4,1-1e-4), t)

        opt.zero_grad(set_to_none=True)
        L.backward()
        opt.step()

        # EMA teacher update
        with torch.no_grad():
            for tp, sp in zip(teacher.parameters(), m.parameters()):
                tp.data.mul_(ema).add_(sp.data*(1-ema))

    return m.eval()

@torch.no_grad()
def eval_model(m, name):
    Ps, Ys = [], []
    t0=time.time()
    for pils, y, _ in tqdm(test_loader, desc=f"EVAL {name}"):
        x = batch_tf(pils, eval_tf).to(DEVICE)
        probs = torch.sigmoid(m(x)).detach().cpu().numpy()
        Ps.append(probs); Ys.append(y.numpy())
    sec=time.time()-t0

    P=np.vstack(Ps); Y=np.vstack(Ys)
    df, macro = metrics(Y,P)
    ece = expected_calibration_error(Y,P)
    ent = avg_entropy(P)
    print("\n", df); print("MacroAUC:", macro, "| ECE:", ece, "| Entropy:", ent, "| sec:", sec)
    return {"name":name, "macro_auc":macro, "ece":ece, "entropy":ent, "sec":sec, "perclass":df}

# Run classical ablations
adabn_net = adabn(net, stream_loader, n_batches=STREAM_BATCHES)
tent_net  = tent(net,  stream_loader, n_batches=STREAM_BATCHES, lr=TENT_LR)
cotta_net = cotta_lite(net, stream_loader, n_batches=STREAM_BATCHES, lr=COTTA_LR)

adabn_res = eval_model(adabn_net, "AdaBN")
tent_res  = eval_model(tent_net,  "TENT")
cotta_res = eval_model(cotta_net, "CoTTA-lite")


# ✅ PQC TTDA ablations (3 strategies) aligned with PDF loss form

In [ ]:
# =========================================
# CELL 12 — PQC BUILD (QNN + parser dt→{depth,θ,ϕ,α} + λ1,λ2)
# =========================================
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

def build_qnn(nq=6, n_layers=2):
    x = ParameterVector("x", 2*nq + 2)   # [feat_angles(nq), dt_angles(nq), depth_g, alpha]
    w = ParameterVector("w", n_layers * (3*nq))  # trainable

    qc = QuantumCircuit(nq)
    # encode feat angles and dt angles with Ry/Rz
    for i in range(nq):
        qc.ry(x[i], i)
        qc.rz(x[nq+i], i)

    depth_g = x[2*nq + 0]
    alpha   = x[2*nq + 1]

    # entangling blocks: CNOT, CZ, CRZ; dt-conditioned controlled-phase via CRZ(alpha)
    wi = 0
    for L in range(n_layers):
        # rotations (scaled by depth_g to mimic "depth modulation")
        for i in range(nq):
            qc.ry(depth_g * w[wi+0], i); wi += 1
            qc.rz(depth_g * w[wi+0], i); wi += 1
            qc.ry(depth_g * w[wi+0], i); wi += 1

        # entanglement: CNOT chain
        for i in range(nq-1):
            qc.cx(i, i+1)
        # add CZ ring
        qc.cz(nq-1, 0)
        # LLM-conditioned controlled-phase-ish: CRZ(alpha) on some pairs
        for i in range(0, nq-1, 2):
            qc.crz(alpha, i, i+1)

    obs = SparsePauliOp.from_list([("Z" + "I"*(nq-1), 1.0)])  # Z on qubit0
    qnn = EstimatorQNN(
        circuit=qc,
        estimator=Estimator(),
        input_params=list(x),
        weight_params=list(w),
        observables=obs
    )
    return qnn

class DtParser(nn.Module):
    # dt -> depth_g, alpha, lam1, lam2, dt_angles
    def __init__(self, dt_dim=768, nq=6):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dt_dim, 256),
            nn.GELU(),
            nn.Linear(256, 128),
            nn.GELU(),
        )
        self.to_dt_angles = nn.Linear(128, nq)
        self.to_depth = nn.Linear(128, 1)
        self.to_alpha = nn.Linear(128, 1)
        self.to_lams  = nn.Linear(128, 2)

    def forward(self, dt):
        h = self.fc(dt)
        dt_angles = torch.tanh(self.to_dt_angles(h)) * math.pi
        depth_g   = torch.sigmoid(self.to_depth(h))          # (B,1)
        alpha     = torch.tanh(self.to_alpha(h)) * math.pi   # (B,1)
        lams      = torch.sigmoid(self.to_lams(h))           # (B,2) -> λ1,λ2
        lam1, lam2 = lams[:,0:1], lams[:,1:2]
        return dt_angles, depth_g, alpha, lam1, lam2

class PQCResidualHead(nn.Module):
    def __init__(self, nq=6, dt_dim=768, out_dim=5):
        super().__init__()
        self.nq = nq
        self.dt_parser = DtParser(dt_dim=dt_dim, nq=nq)

        self.feat_proj = nn.Linear(2048, nq)  # CNN feat -> angles
        self.qnn = TorchConnector(build_qnn(nq=nq, n_layers=N_ENT_LAYERS)).to("cpu")

        self.q_mean = nn.Parameter(torch.zeros(1), requires_grad=False)

        # lightweight MLP classifier (PDF)
        self.mlp = nn.Sequential(
            nn.Linear(1, 64),
            nn.GELU(),
            nn.Linear(64, out_dim)
        )
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, feats_cpu, dt_cpu):
        # feats_cpu: (B,2048) on cpu
        feat_angles = torch.tanh(self.feat_proj(feats_cpu)) * math.pi  # (B,nq)

        dt_angles, depth_g, alpha, lam1, lam2 = self.dt_parser(dt_cpu) # cpu tensors
        q_in = torch.cat([feat_angles, dt_angles, depth_g, alpha], dim=1).to(torch.float64)  # (B,2nq+2)
        q_out = self.qnn(q_in).to(torch.float32)  # (B,1)

        q_center = q_out - self.q_mean
        delta = self.mlp(q_center)  # (B,C)
        return delta, q_out, lam1, lam2


In [ ]:
# =========================================
# CELL 13 — PQC MEMORY (Cell 8-like prototypes) + TTDA losses
# =========================================
def cosine_fid(q_out, q_ref):
    # q_out: (B,1) , q_ref: (1,)
    q = q_out.view(-1)
    ref = q_ref.view(-1)
    # scalar cosine -> use L2 distance as stable proxy
    return ((q - ref)**2).mean()

def build_source_memory(head, max_batches=None):
    """
    This matches your screenshot:
    - run over labeled test set (or any labeled set)
    - build per-class prototype vectors (mean q_out) + counts
    """
    head.eval()
    proto = torch.zeros(C, 1)
    counts = torch.zeros(C)

    t0 = time.time()
    for bi, (pils, y, metas) in enumerate(tqdm(test_loader, desc="Build SourceMemory")):
        if max_batches is not None and bi >= max_batches:
            break
        feats, _ = forward_feats_logits(pils, eval_tf)
        dt = get_dt_for_batch(metas, prefer="target")   # cpu (B,768)

        with torch.no_grad():
            _, q_out, _, _ = head(feats.float(), dt.float())  # cpu

        y_np = y.numpy()  # (B,C)
        for j in range(C):
            pos = (y_np[:,j] > 0.5)
            if pos.sum() == 0:
                continue
            qj = q_out[pos].mean(dim=0, keepdim=True)
            proto[j] = proto[j] + qj
            counts[j] += float(pos.sum())

    # finalize mean
    for j in range(C):
        if counts[j] > 0:
            proto[j] = proto[j] / counts[j]

    print("✅ SourceMemory prototypes built | counts:", counts.tolist(), "| sec:", time.time()-t0)
    return proto.clone(), counts.clone()

def update_target_memory(tgt_proto, tgt_counts, q_out, y_hat=None, y_true=None):
    """
    Target memory update happens DURING TTDA (not training).
    If you have target labels, use y_true; else use confident pseudo y_hat.
    """
    if y_true is not None:
        y_use = y_true
    else:
        y_use = (y_hat > 0.5).float()

    with torch.no_grad():
        for j in range(C):
            pos = (y_use[:,j] > 0.5)
            if pos.sum() == 0:
                continue
            qj = q_out[pos].mean(dim=0, keepdim=True)  # (1,1)
            tgt_proto[j] = (1-MEM_LR)*tgt_proto[j] + MEM_LR*qj
            tgt_counts[j] += float(pos.sum())


In [ ]:
# =========================================
# CELL 14 — PQC TTDA STRATEGIES (3 ablations) + eval
# =========================================
def pqc_entropy_loss(student_logits):
    return entropy_loss_from_logits(student_logits)

def pqc_distill_loss(student_logits, metas):
    if not USE_TEACHER_DISTILL:
        return 0.0
    t = get_teacher_batch(metas, device="cpu")
    if t is None:
        return 0.0
    s = torch.sigmoid(student_logits).clamp(1e-4,1-1e-4)
    return BETA_DISTILL * F.binary_cross_entropy(s, t)

@torch.no_grad()
def eval_pqc(head, name):
    Ps, Ys = [], []
    t0 = time.time()
    for pils, y, metas in tqdm(test_loader, desc=f"EVAL {name}"):
        feats, base_logits = forward_feats_logits(pils, eval_tf)
        dt = get_dt_for_batch(metas, prefer="target")
        delta, q_out, _, _ = head(feats.float(), dt.float())
        logits = base_logits + (PQC_LOGIT_SCALE * delta)
        Ps.append(torch.sigmoid(logits).numpy()); Ys.append(y.numpy())
    sec=time.time()-t0

    P=np.vstack(Ps); Y=np.vstack(Ys)
    df, macro = metrics(Y,P)
    ece = expected_calibration_error(Y,P)
    ent = avg_entropy(P)
    print("\n", df); print("MacroAUC:", macro, "| ECE:", ece, "| Entropy:", ent, "| sec:", sec)
    return {"name":name, "macro_auc":macro, "ece":ece, "entropy":ent, "sec":sec, "perclass":df}

def run_pqc_ttda(strategy_name, head_init, adapt_steps=250):
    """
    3 strategies:
      - PQC-TENT     : entropy + fidelity + optional distill
      - PQC-EATA-lite: entropy on reliable (low entropy) samples + fidelity
      - PQC-CoTTA    : EMA teacher consistency + entropy + fidelity
    """
    head = copy.deepcopy(head_init).to("cpu")
    head.train()

    # init q_mean from a few stream batches (source reference)
    qs=[]
    it = iter(stream_loader)
    for _ in range(20):
        pils, _, metas = next(it)
        feats, _ = forward_feats_logits(pils, eval_tf)
        dt = get_dt_for_batch(metas, prefer="target")
        _, q_out, _, _ = head(feats.float(), dt.float())
        qs.append(q_out.detach())
    head.q_mean.copy_(torch.cat(qs, dim=0).mean())

    # build SourceMemory prototypes (like your screenshot)
    src_proto, src_counts = build_source_memory(head)

    # TargetMemory init = SourceMemory (common, stable)
    tgt_proto = src_proto.clone()
    tgt_counts = torch.zeros_like(src_counts)

    # optimizer
    for p in head.parameters():
        p.requires_grad_(True)
    opt = torch.optim.Adam(head.parameters(), lr=LR_PQC)

    teacher = copy.deepcopy(head).eval() if "CoTTA" in strategy_name else None

    log = {"L_total":[], "fid_value":[]}

    it = iter(stream_loader)
    for step in tqdm(range(adapt_steps), desc=f"{strategy_name} TTDA"):
        pils, _, metas = next(it)
        feats, base_logits = forward_feats_logits(pils, eval_tf)  # cpu
        dt = get_dt_for_batch(metas, prefer="target")             # cpu

        delta, q_out, lam1, lam2 = head(feats.float(), dt.float())
        student_logits = base_logits + (PQC_LOGIT_SCALE * delta)

        # entropy
        L_ent = pqc_entropy_loss(student_logits)

        # fidelity regularization (PDF)
        L_fid = cosine_fid(q_out, head.q_mean)

        # LLM-guided weighting (PDF)
        L_total = (lam1.mean() * L_ent) + (lam2.mean() * FID_WEIGHT * L_fid)

        # optional distill (from your CSV)
        L_total = L_total + pqc_distill_loss(student_logits, metas)

        # EATA-lite: only on reliable samples
        if "EATA" in strategy_name:
            with torch.no_grad():
                p = torch.sigmoid(student_logits).clamp(1e-6, 1-1e-6).numpy()
                H = -(p*np.log(p) + (1-p)*np.log(1-p)).mean(axis=1)  # (B,)
                keep = (H < ENT_THR)
            if keep.sum() >= 1:
                # recompute on filtered batch (cheap)
                keep_idx = torch.tensor(np.where(keep)[0], dtype=torch.long)
                delta_k = delta[keep_idx]
                q_out_k = q_out[keep_idx]
                lam1_k = lam1[keep_idx]
                lam2_k = lam2[keep_idx]
                base_k = base_logits[keep_idx]
                logits_k = base_k + (PQC_LOGIT_SCALE * delta_k)
                L_ent = pqc_entropy_loss(logits_k)
                L_fid = cosine_fid(q_out_k, head.q_mean)
                L_total = (lam1_k.mean()*L_ent) + (lam2_k.mean()*FID_WEIGHT*L_fid)
                L_total = L_total + pqc_distill_loss(logits_k, [metas[i] for i in keep_idx.tolist()])

        # CoTTA-lite: teacher consistency
        if teacher is not None:
            with torch.no_grad():
                # same inputs into teacher head
                d_t, q_t, _, _ = teacher(feats.float(), dt.float())
                t_logits = base_logits + (PQC_LOGIT_SCALE * d_t)
            L_total = L_total + 0.5*F.mse_loss(torch.sigmoid(student_logits), torch.sigmoid(t_logits))

        # Memory update + memory contrast (dual memory)
        if USE_MEMORY:
            # pseudo labels from student
            y_hat = torch.sigmoid(student_logits).detach()
            update_target_memory(tgt_proto, tgt_counts, q_out.detach(), y_hat=y_hat)

            # contrast: pull q_out toward source prototypes of predicted positives
            with torch.no_grad():
                y_bin = (y_hat > 0.5).float()  # (B,C)
            pulls=[]
            for j in range(C):
                pos = (y_bin[:,j] > 0.5)
                if pos.sum()==0: 
                    continue
                pulls.append((q_out[pos] - src_proto[j]).pow(2).mean())
            if len(pulls)>0:
                L_total = L_total + 0.20*torch.stack(pulls).mean()

        opt.zero_grad(set_to_none=True)
        L_total.backward()
        opt.step()

        # CoTTA EMA update
        if teacher is not None:
            with torch.no_grad():
                ema=0.999
                for tp, sp in zip(teacher.parameters(), head.parameters()):
                    tp.data.mul_(ema).add_(sp.data*(1-ema))

        log["L_total"].append(float(L_total.detach().cpu()))
        log["fid_value"].append(float(L_fid.detach().cpu()))

    head.eval()
    return head, log


In [ ]:
# =========================================
# CELL 15 — RUN PQC ABLATIONS (3) + FINAL ONE TABLE
# =========================================
# initialize head once (for fair reset)
head0 = PQCResidualHead(nq=NQ, dt_dim=768, out_dim=C)

pqc_tent_head, log_tent = run_pqc_ttda("PQC-TENT", head0, adapt_steps=MAX_PQC_STEPS)
pqc_eata_head, log_eata = run_pqc_ttda("PQC-EATA-lite", head0, adapt_steps=MAX_PQC_STEPS)
pqc_cotta_head, log_cotta = run_pqc_ttda("PQC-CoTTA-lite", head0, adapt_steps=MAX_PQC_STEPS)

pqc_tent_res  = eval_pqc(pqc_tent_head,  "PQC-TENT")
pqc_eata_res  = eval_pqc(pqc_eata_head,  "PQC-EATA-lite")
pqc_cotta_res = eval_pqc(pqc_cotta_head, "PQC-CoTTA-lite")

all_res = [base_res, adabn_res, tent_res, cotta_res, pqc_tent_res, pqc_eata_res, pqc_cotta_res]

summary = pd.DataFrame([{
    "Strategy": r["name"],
    "MacroAUC": r["macro_auc"],
    "ECE": r["ece"],
    "Entropy": r["entropy"],
    "Sec": r["sec"],
} for r in all_res])

print("\n============================")
print("✅ FINAL ABLATION TABLE")
print("============================")
display(summary)


# RAG-STYLE MEMORY PREPROCESS (Stage 0 → Stage 2)

In [3]:
# ============================================================
# RAG-STYLE MEMORY PREPROCESS (Stage 0 → Stage 2)
# Input:  chex_train_features.csv
# Output:
#   memory/
#     embeddings.npy        (ei: normalized keys for retrieval)
#     labels.npy            (yi: multi-label vectors)
#     v_img.npy             (vi: optional extra)
#     t_desc.npy            (ti: optional extra)
#     base_conf.npy         (optional extra)
#     metadata.jsonl        (optional, one JSON per row)
#     index.faiss           (if faiss available)
#     index_sklearn.joblib  (fallback if faiss not available)
#     prototypes.npy        (Stage 2: per-label prototypes)
#     prototypes_labels.json
# ============================================================

import os, json
import numpy as np
import pandas as pd

CSV_PATH = "/kaggle/input/llm-files/chex_train.csv"
OUT_DIR  = "memory"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------
# Load CSV
# -------------------------------
df = pd.read_csv(CSV_PATH)
print("Loaded:", CSV_PATH)
print("Shape :", df.shape)

# -------------------------------
# Stage 0: Define fields
# -------------------------------
# Metadata (mi): keep for debugging + analysis (NOT in key by default)
meta_cols = [
    "dataset", "image", "patient_id",
    "age_meta", "sex_meta", "view_meta", "ap_pa_meta"
]
meta_cols = [c for c in meta_cols if c in df.columns]

# Labels (yi): multi-label
label_cols = [c for c in df.columns if c.startswith("y_")]
if len(label_cols) == 0:
    raise ValueError("No label columns found (expected columns starting with 'y_').")

# Image-side vector vi: cnn_* columns
v_cols = [c for c in df.columns if c.startswith("cnn_")]
if len(v_cols) == 0:
    raise ValueError("No image feature columns found (expected columns starting with 'cnn_').")

# Text/descriptor-side vector ti: q_* columns
t_cols = [c for c in df.columns if c.startswith("q_")]
if len(t_cols) == 0:
    raise ValueError("No text/descriptor feature columns found (expected columns starting with 'q_').")

print("\nColumns used:")
print("  metadata :", meta_cols)
print("  labels   :", label_cols)
print("  v_i cols :", v_cols)
print("  t_i cols :", t_cols)

# -------------------------------
# Stage 0.3: Extract vi, ti, fuse zi=[vi;ti]
# -------------------------------
V = df[v_cols].to_numpy(dtype=np.float32)     # vi
T = df[t_cols].to_numpy(dtype=np.float32)     # ti
Z = np.concatenate([V, T], axis=1).astype(np.float32)  # zi

# -------------------------------
# Stage 0.4: Normalize => ei = zi / ||zi||2
# -------------------------------
eps = 1e-12
norms = np.linalg.norm(Z, axis=1, keepdims=True)
E = Z / (norms + eps)   # ei

# -------------------------------
# Stage 0.5: Sanity checks
# -------------------------------
# Check NaNs / inf
if not np.isfinite(E).all():
    bad = np.where(~np.isfinite(E).all(axis=1))[0][:10]
    raise ValueError(f"Found non-finite values in embeddings. Example bad rows: {bad.tolist()}")

# Check norms ~ 1 for 1000 samples
n_check = min(1000, E.shape[0])
check_idx = np.random.choice(E.shape[0], size=n_check, replace=False)
check_norms = np.linalg.norm(E[check_idx], axis=1)

print("\nSanity checks (Stage 0.5):")
print(f"  Norm mean/std (n={n_check}): {check_norms.mean():.6f} / {check_norms.std():.6f}")
print(f"  Norm min/max            : {check_norms.min():.6f} / {check_norms.max():.6f}")
print(f"  Embedding mean/std (all): {E.mean():.6f} / {E.std():.6f}")

# Labels matrix (multi-label)
Y = df[label_cols].to_numpy(dtype=np.int64)

# Optional baseline confidence (recommended extra field)
# (here: max of cnn_* features; if you have real baseline probs, use those instead)
base_conf = df[v_cols].max(axis=1).to_numpy(dtype=np.float32)

# -------------------------------
# Stage 1: Store memory (minimum + recommended extras)
#   Minimum:
#     - Key: ei  (E)
#     - Value: yi (Y)
#   Recommended extras:
#     - vi (V), ti (T), baseline confidence, metadata
# -------------------------------
np.save(os.path.join(OUT_DIR, "embeddings.npy"), E)   # ei
np.save(os.path.join(OUT_DIR, "labels.npy"), Y)       # yi
np.save(os.path.join(OUT_DIR, "v_img.npy"), V)        # vi (extra)
np.save(os.path.join(OUT_DIR, "t_desc.npy"), T)       # ti (extra)
np.save(os.path.join(OUT_DIR, "base_conf.npy"), base_conf)

# ---- FIX: JSONL writer that handles numpy/pandas scalar types safely ----
def _json_default(o):
    # numpy scalars (np.int64, np.float32, etc.)
    if isinstance(o, np.generic):
        return o.item()
    # numpy arrays
    if isinstance(o, np.ndarray):
        return o.tolist()
    # pandas missing values / NaT
    if o is pd.NA:
        return None
    # pandas Timestamp
    if isinstance(o, pd.Timestamp):
        return o.isoformat()
    # last resort: stringify
    return str(o)

meta_path = os.path.join(OUT_DIR, "metadata.jsonl")
with open(meta_path, "w", encoding="utf-8") as f:
    for i in range(len(df)):
        rec = {c: df.at[i, c] for c in meta_cols}
        f.write(json.dumps(rec, ensure_ascii=False, default=_json_default) + "\n")

print("\nSaved memory files to:", OUT_DIR)

# -------------------------------
# Stage 2: Choose Memory Granularity (Instance vs Prototype)
#   2.1 Instance Memory: already done (every sample stored)
#   2.2 Prototype Memory (per-label prototype for multi-label):
#       For each label ℓ: μ_ℓ = mean(ei over samples with y_ℓ = 1)
# -------------------------------
prototypes = []
proto_names = []
for j, lab in enumerate(label_cols):
    mask = (Y[:, j] == 1)
    if mask.sum() == 0:
        print("Warning: no positive samples for", lab, "-> skipping prototype.")
        continue
    mu = E[mask].mean(axis=0)
    mu = mu / (np.linalg.norm(mu) + eps)  # normalize prototype too
    prototypes.append(mu.astype(np.float32))
    proto_names.append(lab)

if len(prototypes) > 0:
    prototypes = np.stack(prototypes, axis=0)
else:
    prototypes = np.zeros((0, E.shape[1]), dtype=np.float32)

np.save(os.path.join(OUT_DIR, "prototypes.npy"), prototypes)
with open(os.path.join(OUT_DIR, "prototypes_labels.json"), "w", encoding="utf-8") as f:
    json.dump(proto_names, f, ensure_ascii=False, indent=2)

print(f"\nPrototype memory saved: prototypes.npy with shape {prototypes.shape}")

# -------------------------------
# Stage 3 (optional but recommended by PDF): Build retrieval index
#   Prefer FAISS IndexFlatIP (inner product).
#   With normalized vectors, inner product == cosine similarity ranking.
# -------------------------------
index_built = False
try:
    import faiss  # pip install faiss-cpu
    d = E.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(E.astype(np.float32))
    faiss.write_index(index, os.path.join(OUT_DIR, "index.faiss"))
    print("\nFAISS index built: memory/index.faiss")
    index_built = True

    # Retrieval sanity check: self-retrieval on a few points
    Q = E[:10].astype(np.float32)
    sims, idxs = index.search(Q, 1)
    print("Self-retrieval check (top-1 indices):", idxs.reshape(-1).tolist())

except Exception as e:
    print("\nFAISS not available or failed to build index:", repr(e))
    print("Falling back to scikit-learn NearestNeighbors (cosine).")

if not index_built:
    from sklearn.neighbors import NearestNeighbors
    import joblib

    nn = NearestNeighbors(n_neighbors=10, metric="cosine", algorithm="brute")
    nn.fit(E)

    joblib.dump(nn, os.path.join(OUT_DIR, "index_sklearn.joblib"))
    print("sklearn NN index built: memory/index_sklearn.joblib")

    dists, idxs = nn.kneighbors(E[:10], n_neighbors=1)
    print("Self-retrieval check (top-1 indices):", idxs.reshape(-1).tolist())

print("\nDONE ✅ Your RAG-style memory is ready.")
print("Key (ei) shape:", E.shape, "| Labels (yi) shape:", Y.shape)


Loaded: /kaggle/input/llm-files/chex_train.csv
Shape : (30000, 25)

Columns used:
  metadata : ['dataset', 'image', 'patient_id', 'age_meta', 'sex_meta', 'view_meta', 'ap_pa_meta']
  labels   : ['y_Atelectasis', 'y_Cardiomegaly', 'y_Consolidation', 'y_Edema', 'y_Pleural Effusion']
  v_i cols : ['cnn_Atelectasis', 'cnn_Cardiomegaly', 'cnn_Consolidation', 'cnn_Edema', 'cnn_Pleural Effusion']
  t_i cols : ['q_mean', 'q_std', 'q_entropy', 'q_sharp_lapvar', 'q_edge_density', 'q_clip_low', 'q_clip_high']

Sanity checks (Stage 0.5):
  Norm mean/std (n=1000): 1.000000 / 0.000000
  Norm min/max            : 1.000000 / 1.000000
  Embedding mean/std (all): 0.114781 / 0.264875

Saved memory files to: memory

Prototype memory saved: prototypes.npy with shape (5, 12)

FAISS not available or failed to build index: ModuleNotFoundError("No module named 'faiss'")
Falling back to scikit-learn NearestNeighbors (cosine).
sklearn NN index built: memory/index_sklearn.joblib
Self-retrieval check (top-1 indice